In [ ]:
# define Meta data

# rec_loops = 3



### Paper : Less is More: Recursive Reasoning with Tiny Networks, Oct 2025
(https://arxiv.org/pdf/2510.04871)

## Problem Overview

We study Recursive Reasoning using Tiny Networks (TRM).

Goal:
Instead of increasing model size, we improve reasoning by:
- reusing the same network multiple times
- learning when to halt (q_halt)

Key Idea:
Compute multiple reasoning steps and decide dynamically when to stop.

In [ ]:
import torch
from torch import nn

## Model Architecture

Components:
- f: prediction head
- q_halt: halting probability

At each step t:
1. Model updates hidden state
2. Predicts output
3. Predicts whether to stop

Final output depends on halting decision.

- Latent layers are not directly tracked in gradients.
- They are used to refine the internal states.
- This helps y and z move toward a stable representation before reasoning begins.

In [ ]:
class TinyRModel(nn.Module):
  def __init__(self, hidden_size, vocab_size, inp_seq_len, out_seq_len):
    super().__init__()
    self.net = nn.Sequential(nn.Linear(hidden_size, hidden_size),
                              nn.ReLU(),
                              nn.Linear(hidden_size, hidden_size))

    self.embedding = nn.Embedding(vocab_size, hidden_size)
    self.input_proj = nn.Linear(inp_seq_len * hidden_size, hidden_size)


    self.out = nn.Linear(hidden_size, out_seq_len * vocab_size)
    self.halt = nn.Linear(hidden_size, 1)



  def latent(self, x, y, z, n=6):
      # Phase 1: update z (n times), x included
      for i in range(n):
          z = self.net(x + y + z)   # element-wise add → D-dim input

      # Phase 2: update y once, WITHOUT x
      y = self.net(y + z)           # same net, different input mix

      return y, z



  def forward(self, x, y, z, T, n=6):

    x = self.embedding(x)          # (B, L, H)
    x = x.view(x.size(0), -1)              # flatten
    x = self.input_proj(x)                 # (B, H)

    with torch.no_grad():
      for j in range(T-1):
        y, z= self.latent(x,y,z,n)

    y, z = self.latent(x, y, z, n)

    f_out = self.out(y)

    halt= self.halt(y) #calculate halt from y

    return f_out, halt, y, z

### Dataset Description: Majority Task
The dataset consists of fixed-length binary sequences ($0$s and $1$s). The goal is to classify the sequence based on the majority bit:
- **Label 1**: If the count of $1$s is greater than the count of $0$s.
- **Label 0**: If the count of $0$s is greater than the count of $1$s.

Ties are excluded to ensure unambiguous labeling. This serves as a simple reasoning task for the recursive model.

In [ ]:
# Simple dataset

# import torch
# import random
# from torch.utils.data import Dataset, DataLoader, random_split

# # ── config ───────────────────────────────────────────────────────────────
# hidden_size = 64
# batch_size  = 32
# device      = 'cuda' if torch.cuda.is_available() else 'cpu'
# vocab_size  = 2    # only real tokens: 0 and 1
# num_classes = 2    # label 0 or 1
# seq_len     = 15   # fixed length — no padding needed

# # ── dataset ──────────────────────────────────────────────────────────────
# class MajorityDataset(Dataset):
#     """
#     Fixed-length binary sequences, no padding, no masking.
#     Label = 1 if 1s are majority, 0 if 0s are majority.
#     Ties skipped to avoid ambiguous labels.
#     """
#     def __init__(self, num_samples=10000, seq_len=seq_len):
#         self.samples = self._generate(num_samples, seq_len)

#     def _generate(self, n, seq_len):
#         samples = []
#         while len(samples) < n:
#             seq   = [random.randint(0, 1) for _ in range(seq_len)]
#             ones  = sum(seq)
#             zeros = seq_len - ones
#             if ones == zeros:       # skip ties
#                 continue
#             label = 1 if ones > zeros else 0
#             samples.append((seq, label))
#         return samples

#     def __len__(self):
#         return len(self.samples)

#     def __getitem__(self, idx):
#         seq, label = self.samples[idx]
#         return {
#             "input_ids": torch.tensor(seq,   dtype=torch.long),
#             "labels":    torch.tensor(label, dtype=torch.long),
#         }

# # ── dataloaders ──────────────────────────────────────────────────────────
# dataset  = MajorityDataset(num_samples=10000)
# val_size = 1000
# train_set, val_set = random_split(dataset, [len(dataset) - val_size, val_size])

# train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
# val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)

## Addition Dataset

In [ ]:
import torch
import random
from torch.utils.data import Dataset, DataLoader, random_split

def generate_pair(num_digits):
    a = random.randint(10**(num_digits-1), 10**num_digits - 1)
    b = random.randint(10**(num_digits-1), 10**num_digits - 1)
    return str(a), str(b)


operands_a, operands_b = [], []

for sample_idx in range(50000):
    num_a, num_b = generate_pair(num_digits=4)

    operands_a.append(num_a)
    operands_b.append(num_b)

In [ ]:
dataset = []

for num_a, num_b in zip(operands_a, operands_b):

    input_sequence = list(num_a) + ['+'] + list(num_b)

    rev_a = list(reversed(num_a))
    rev_b = list(reversed(num_b))

    num_digits = len(num_a)

    result_digits = []
    carry_sequence = []

    carry = 0
    current_chain_length = 0
    max_carry_chain = 0

    for digit_a, digit_b in zip(rev_a, rev_b):

        digit_sum = int(digit_a) + int(digit_b) + carry

        output_digit = digit_sum % 10
        new_carry = digit_sum // 10

        result_digits.append(output_digit)
        carry_sequence.append(new_carry)

        if new_carry == 1:
            current_chain_length += 1
            max_carry_chain = max(max_carry_chain, current_chain_length)
        else:
            current_chain_length = 0

        carry = new_carry

    if carry > 0:
        result_digits.append(carry)

    final_output = list(reversed(result_digits))

    data = {
        'input': input_sequence,
        'output': final_output,
        'num_digits': num_digits,
        'carry_sequence': carry_sequence,
        'max_carry_chain': max_carry_chain
    }

    dataset.append(data)

print(dataset[0], len(dataset))

{'input': ['7', '1', '0', '2', '+', '9', '2', '5', '6'], 'output': [1, 6, 3, 5, 8], 'num_digits': 4, 'carry_sequence': [0, 0, 0, 1], 'max_carry_chain': 1} 50000


In [ ]:
import torch
from torch.utils.data import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'

PAD_TOKEN = 11
PLUS_TOKEN = 10

inp_seq_len = 9
out_seq_len = 5


class AdditionDataset(Dataset):
    def __init__(self, dataset_list, max_input_len=inp_seq_len, max_output_len=out_seq_len):
        self.data = dataset_list
        self.max_input_len = max_input_len
        self.max_output_len = max_output_len

    def pad(self, seq, max_len, pad_value=PAD_TOKEN):
        return seq + [pad_value] * (max_len - len(seq))

    def encode_input(self, input_seq):
        return [
            int(x) if x != '+' else PLUS_TOKEN
            for x in input_seq
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]

        input_ids = self.encode_input(sample['input'])
        labels    = sample['output']

        input_ids = self.pad(input_ids, self.max_input_len)
        labels    = self.pad(labels, self.max_output_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long)
        }



hidden_size = 256

batch_size  = 32

vocab_size = 12

dataset = AdditionDataset(dataset)

val_size = 5000
train_size = len(dataset) - val_size

train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=batch_size)

## Loss Function

We use two losses:

1. Prediction Loss:
   Cross entropy between prediction and label

2. Halting Loss:
   Binary classification:
   - 1 if prediction correct
   - 0 otherwise

This encourages:
→ stop when correct
→ continue when wrong

In [ ]:
# def Q_LOSS(q_halt, f_out, y_true):
#     print(f_out.argmax(dim=-1), y_true)
#     target = f_out.argmax(dim=-1) == y_true
#     print(target)
#     return F.binary_cross_entropy_with_logits(q_halt, target)

In [ ]:
# def model_training_and_validation_with_mask(c, T, n_sup):
#     model = TinyRModel(hidden_size, vocab_size, inp_seq_len=12, out_seq_len=6).to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#     print('PAD_TOKEN: ', PAD_TOKEN)
#     criterion = nn.CrossEntropyLoss(reduction='none', ignore_index=PAD_TOKEN)

#     running_loss = 0
#     running_q = 0

#     print('Training Logs ------ \n')
#     model.train()
#     for step, batch in enumerate(train_loader):
#         input_ids = batch['input_ids'].to(device)
#         labels = batch['labels'].to(device)
#         batch_size_curr = input_ids.size(0)
#         seq_len = labels.size(1)

#         active_mask = torch.ones(batch_size_curr, dtype=torch.bool, device=device)

#         x = input_ids
#         y = torch.zeros(batch_size_curr, hidden_size, device=device)
#         z = torch.zeros(batch_size_curr, hidden_size, device=device)

#         step_loss = step_q = 0
#         overall_loss = torch.tensor(0.0, device=device)

#         for i in range(n_sup):
#             y_new, z_new, halt, out = model(x, y, z, halt, out)


#             y = torch.where(active_mask.unsqueeze(1), y_new.detach(), y)
#             z = torch.where(active_mask.unsqueeze(1), z_new.detach(), y)

#             loss_per_token = crossntropy(f_out.view(-1, vocab_size), labels.view(-1))




In [ ]:
import torch.nn.functional as F

def Q_LOSS(q_halt, f_out, y_true, pad_token=PAD_TOKEN):
    logp = F.log_softmax(f_out, dim=-1)                                        # (B, L, V)
    logp_correct = logp.gather(-1, y_true.unsqueeze(-1)).squeeze(-1)           # (B, L): log p(correct token)
    non_pad = (y_true != pad_token).float()                                    # (B, L)
    mean_logp = (logp_correct * non_pad).sum(dim=-1, keepdim=True) / (non_pad.sum(dim=-1, keepdim=True) + 1e-8)
    target = mean_logp.exp().detach()                                          # (B, 1): geo-mean of correct-token probs = exp(-mean CE)
    return F.binary_cross_entropy_with_logits(q_halt, target)

In [ ]:
### claude generate this diagnostic script but i guided it what to do


from torch.utils.data import Subset


def diagnostic_per_sup_step(c=0.1, T=3, n_sup=16, num_epochs=100):
    model = TinyRModel(hidden_size, vocab_size, inp_seq_len = inp_seq_len, out_seq_len = out_seq_len).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion_noreduce = nn.CrossEntropyLoss(reduction='none', ignore_index=PAD_TOKEN)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

    diag_subset = Subset(train_set, range(max(10000, len(train_set))))
    diag_loader = DataLoader(diag_subset, batch_size=64, shuffle=True)

    model.train()

    for epoch in range(num_epochs):
        print(f'processing epoch {epoch} \n')
        for batch in diag_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            batch_size_curr = input_ids.size(0)
            seq_len = labels.size(1)

            x = input_ids
            y = torch.zeros(batch_size_curr, hidden_size, device=device)
            z = torch.zeros(batch_size_curr, hidden_size, device=device)

            total_loss = torch.tensor(0.0, device=device)

            for i in range(n_sup):
                f_out, halt, y_new, z_new = model(x, y, z, T, n=6)
                y = y_new.detach()
                z = z_new.detach()

                loss = criterion(f_out.view(-1, vocab_size), labels.view(-1))

                # Add Q-loss with the SAME formula as masked run
                seq_len = labels.size(1)
                q_loss = Q_LOSS(halt, f_out.view(-1, seq_len, vocab_size), labels)

                total_loss += (loss + c * q_loss.squeeze())  # same coefficient as masked run

            total_loss /= n_sup

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()

        # --- Detailed per-step evaluation every 10 epochs ---
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                test_batch = next(iter(diag_loader))
                input_ids = test_batch['input_ids'].to(device)
                labels = test_batch['labels'].to(device)
                batch_size_curr = input_ids.size(0)
                seq_len = labels.size(1)

                x = input_ids
                y_state = torch.zeros(batch_size_curr, hidden_size, device=device)
                z_state = torch.zeros(batch_size_curr, hidden_size, device=device)

                print(f"Epoch {epoch:3d} | CE per supervision step (same batch):")

                prev_ce = None
                for i in range(n_sup):
                    f_out, halt, y_new, z_new = model(x, y_state, z_state, T, n=6)
                    y_state = y_new.detach()
                    z_state = z_new.detach()

                    print(f"sup_step_{i+1} | mean={halt.mean().item():.3f} "
                          f"std={halt.std().item():.3f} "
                          f"median={halt.median().item():.3f} "
                          f"%>0={(halt > 0).float().mean().item():.1%} "
                          f"first5={halt[:5].squeeze().tolist()}")

                    # Per-token CE (ignoring padding)
                    loss_per_token = criterion_noreduce(f_out.view(-1, vocab_size), labels.view(-1))
                    # Reshape to (Batch, Seq)
                    loss_grid = loss_per_token.view(batch_size_curr, seq_len)
                    non_pad = (labels != PAD_TOKEN)
                    # Per-sample CE on real tokens only
                    ce_per_sample = (loss_grid * non_pad.float()).sum(dim=1) / non_pad.float().sum(dim=1).clamp(min=1)
                    avg_ce = ce_per_sample.mean().item()

                    # Token accuracy EXCLUDING padding
                    preds = f_out.view(-1, seq_len, vocab_size).argmax(dim=-1)
                    correct = (preds == labels) & non_pad
                    token_acc = correct.sum().item() / non_pad.sum().item()

                    # Exact match: all NON-PAD positions correct
                    per_sample_correct = ((preds == labels) | ~non_pad).all(dim=-1).float()
                    exact = per_sample_correct.mean().item()

                    # CE change from previous step
                    delta = ""
                    if prev_ce is not None:
                        diff = prev_ce - avg_ce
                        delta = f"| Delta: {diff:+.4f} {'IMPROVED' if diff > 0.01 else 'FLAT' if diff > -0.01 else 'WORSE'}"
                    prev_ce = avg_ce

                    # Prediction diversity
                    pred_strings = [tuple(p.cpu().tolist()) for p in preds]
                    unique_preds = len(set(pred_strings))

                    print(f"  Sup {i:2d} | CE: {avg_ce:.4f} | TokAcc: {token_acc:.4f} | Exact: {exact:.4f} | Unique: {unique_preds}/{batch_size_curr} {delta}")

                print()

                # Show sample predictions at LAST sup step (non-pad only)
                if epoch % 50 == 0:
                    print(f"--- Sample Predictions at Epoch {epoch}, Last Sup Step ---")
                    for j in range(min(10, batch_size_curr)):
                        mask = non_pad[j]
                        pred_real = preds[j][mask].cpu().tolist()
                        true_real = labels[j][mask].cpu().tolist()
                        inp = input_ids[j].cpu().tolist()
                        # Decode input nicely
                        inp_str = ''.join([str(t) if t < 10 else '+' if t == 10 else '' for t in inp])
                        match = "MATCH" if pred_real == true_real else "WRONG"
                        print(f"  {inp_str} | Pred: {pred_real} True: {true_real} {match}")
                    print()

            model.train()

    # --- Final comprehensive breakdown ---
    print("=" * 70)
    print("FINAL BREAKDOWN: All Supervision Steps (Last Epoch)")
    print("=" * 70)

    model.eval()
    with torch.no_grad():
        test_batch = next(iter(diag_loader))
        input_ids = test_batch['input_ids'].to(device)
        labels = test_batch['labels'].to(device)
        batch_size_curr = input_ids.size(0)
        seq_len = labels.size(1)

        x = input_ids
        y_state = torch.zeros(batch_size_curr, hidden_size, device=device)
        z_state = torch.zeros(batch_size_curr, hidden_size, device=device)

        non_pad = (labels != PAD_TOKEN)

        for i in range(n_sup):
            f_out, halt, y_new, z_new = model(x, y_state, z_state, T, n=6)
            y_state = y_new.detach()
            z_state = z_new.detach()

            loss_per_token = criterion_noreduce(f_out.view(-1, vocab_size), labels.view(-1))
            loss_grid = loss_per_token.view(batch_size_curr, seq_len)
            ce_per_sample = (loss_grid * non_pad.float()).sum(dim=1) / non_pad.float().sum(dim=1).clamp(min=1)
            avg_ce = ce_per_sample.mean().item()

            preds = f_out.view(-1, seq_len, vocab_size).argmax(dim=-1)
            correct = (preds == labels) & non_pad
            token_acc = correct.sum().item() / non_pad.sum().item()

            per_sample_correct = ((preds == labels) | ~non_pad).all(dim=-1).float()
            exact = per_sample_correct.mean().item()

            pred_strings = [tuple(p.cpu().tolist()) for p in preds]
            unique_preds = len(set(pred_strings))

            q_mean = halt.mean().item()

            print(f"  Sup {i:2d} | CE: {avg_ce:.4f} | TokAcc: {token_acc:.4f} | Exact: {exact:.4f} | Unique: {unique_preds}/{batch_size_curr} | Q: {q_mean:.4f}")

        # Final sample predictions
        print(f"\n--- Final Sample Predictions ---")
        for j in range(min(10, batch_size_curr)):
            mask = non_pad[j]
            pred_real = preds[j][mask].cpu().tolist()
            true_real = labels[j][mask].cpu().tolist()
            inp = input_ids[j].cpu().tolist()
            inp_str = ''.join([str(t) if t < 10 else '+' if t == 10 else '' for t in inp])
            match = "MATCH" if pred_real == true_real else "WRONG"
            print(f"  {inp_str} | Pred: {pred_real} True: {true_real} {match}")

    return model

model = diagnostic_per_sup_step(c=0.1, T=3, n_sup=16, num_epochs=100)

processing epoch 0 

Epoch   0 | CE per supervision step (same batch):
sup_step_1 | mean=-1.480 std=0.210 median=-1.435 %>0=0.0% first5=[-1.4314385652542114, -1.4264651536941528, -1.188782811164856, -1.4778696298599243, -1.7748421430587769]
  Sup  0 | CE: 1.7987 | TokAcc: 0.3244 | Exact: 0.0000 | Unique: 37/64 
sup_step_2 | mean=-1.462 std=0.158 median=-1.439 %>0=0.0% first5=[-1.4334312677383423, -1.4804795980453491, -1.3248499631881714, -1.5313849449157715, -1.351124882698059]
  Sup  1 | CE: 1.6567 | TokAcc: 0.3645 | Exact: 0.0000 | Unique: 35/64 | Delta: +0.1420 IMPROVED
sup_step_3 | mean=-1.438 std=0.165 median=-1.436 %>0=0.0% first5=[-1.3961395025253296, -1.4916051626205444, -1.1540048122406006, -1.4824432134628296, -1.2598496675491333]
  Sup  2 | CE: 1.6724 | TokAcc: 0.3344 | Exact: 0.0000 | Unique: 33/64 | Delta: -0.0158 WORSE
sup_step_4 | mean=-1.446 std=0.156 median=-1.422 %>0=0.0% first5=[-1.4189404249191284, -1.5044912099838257, -1.28298819065094, -1.4658399820327759, -1.2728

In [ ]:
# ============================================================================
# Q-LOSS AND HALT MECHANISM: STEP-BY-STEP DEBUGGING JOURNEY
# 4-digit addition with TRM, single-head soft-mean Q-loss
# ============================================================================


# ----------------------------------------------------------------------------
# STEP 1: THE INITIAL ILLUSION OF SUCCESS
# ----------------------------------------------------------------------------
# Trained masked run for 100 epochs. CE dropped to ~0.008, Q-loss to ~0.20,
# avg_steps went from 5 down to 1. On the surface this looked like the model
# was learning to halt early once it had the answer figured out.
#
# The framing was: "model got smart, learned it only needs one step,
# adaptive halting working as designed."


# ----------------------------------------------------------------------------
# STEP 2: avg_steps = 1.0 IS NOT EFFICIENT HALTING, IT IS COLLAPSE
# ----------------------------------------------------------------------------
# At epoch 10, val accuracy was only 2.5% but halt was already firing at
# step 1 for 100% of validation samples. By epoch 30, val accuracy was 18%.
# Step-1 halting was happening LONG before the model had learned anything.
#
# This means recursion across supervision steps was never actually used.
# The model learned 4-digit addition entirely through the depth of one
# forward pass (T*n + 1 = 19 net evaluations). The halt head became
# vestigial - always saying "stop now" regardless of correctness.


# ----------------------------------------------------------------------------
# STEP 3: METRIC FIX - avg_steps WAS PER-BATCH-MAX, NOT PER-SAMPLE-MEAN
# ----------------------------------------------------------------------------
# The avg_steps metric was computed using when the loop breaks (when ALL
# samples halt). That gives the per-batch maximum, not the per-sample mean.
#
# Fixed by tracking halt_step per sample, then computing distribution stats:
# mean, median, %step1, %never_halted, full distribution. This gave us the
# real picture - at epoch 10, %step1 was already 88.9%.


# ----------------------------------------------------------------------------
# STEP 4: BUG IN Q_LOSS - PAD TOKENS WERE INFLATING THE TARGET
# ----------------------------------------------------------------------------
# The soft-mean target was computed over ALL positions including pad.
# When the model learned to predict PAD at pad positions, those matches
# counted as "correct", artificially inflating the target value.
#
# Concretely: a 4-digit answer (5 positions, 1 pad) where 2 of 4 real
# digits are right has true token accuracy 2/4 = 0.5, but the soft-mean
# computed it as 3/5 = 0.6, pushing halt logit harder toward firing.
#
# Fixed by masking pad positions in the target computation. But this was
# not the root cause - it was a symptom of the deeper target-design issue.


# ----------------------------------------------------------------------------
# STEP 5: WORKED OUT THE GRADIENT DYNAMICS PRECISELY
# ----------------------------------------------------------------------------
# Q-loss is binary cross-entropy on halt logit h with target t.
# Even though we never call sigmoid in the code, BCE-with-logits applies
# it internally.
#
# The gradient on h is: dL/dh = sigmoid(h) - t
# The optimizer subtracts the gradient, so h moves toward logit(t).
#
# Initial state: h is near 0 (random init), so sigmoid(h) is near 0.5.
# Gradient direction depends on whether target is above or below 0.5:
#
#     t < 0.5: gradient is positive, h decreases, halt does not fire
#     t = 0.5: gradient is zero, h stays put
#     t > 0.5: gradient is negative, h increases, halt fires
#
# The decision rule "halt > 0" is equivalent to "sigmoid(halt) > 0.5"
# because sigmoid is monotonic and sigmoid(0) = 0.5.
#
# So halt fires when target crosses 0.5. With soft-mean target, this means
# halt fires when token accuracy crosses 0.5. But we want halt to fire when
# the WHOLE answer is right (EM = 1.0), not when half the tokens are right.
#
# The semantic mismatch: target measures partial correctness, decision rule
# expects full correctness. They were never aligned.


# ----------------------------------------------------------------------------
# STEP 6: TRIED A "BISTABILITY" FRAMEWORK - FALSIFIED BY DATA
# ----------------------------------------------------------------------------
# Hypothesis: same Q-loss might converge to opposite policies in different
# regimes. With masked training (loop breaks on halt), the Q-head sees
# gradient mostly from early supervision steps. With diagnostic training
# (full rollout), it sees gradient from all 16 steps. Maybe these produce
# opposite equilibria - "always halt" vs "never halt".
#
# Tested by directly logging halt logits at epoch 0 in both regimes:
#     Masked epoch 0:     mean halt = -0.65, %>0 = 0%
#     Diagnostic epoch 0: mean halt = -0.66, %>0 = 0%
#
# Identical starting points. By epoch 10 BOTH regimes had halt logits
# above 0 with 100% firing. No bistability. The driver is target value
# crossing 0.5, not regime-specific gradient distribution.
#
# The simpler story holds: as token accuracy crosses 0.5 across training,
# the equilibrium logit crosses 0, and halt fires regardless of regime.


# ----------------------------------------------------------------------------
# STEP 7: SETTLED ON HYPOTHESIS C (BOTH PROBLEMS, INDEPENDENT)
# ----------------------------------------------------------------------------
#
# A) TASK SIMPLICITY:
#    Diagnostic run (no halt, full rollout) showed Sup 0 -> Sup 1 buys
#    ~19 EM points, but Sup 1 -> Sup 15 is flat. Recursion plateaus
#    after one refinement step. 4-digit addition is a fixed-depth
#    computation - one deep pass plus optionally one refinement is enough.
#
# B) Q-LOSS BROKEN:
#    Halt fires when token accuracy crosses 0.5, far below sequence-level
#    correctness. Step 6 showed this happens regardless of training regime.
#
# C) BOTH ARE TRUE AND THEY ARE INDEPENDENT:
#    On this task, fixing Q-loss alone gets you at most the bounded
#    recursion benefit (~19 EM points from one refinement step).
#    The two failures don't compound - they live in separate parts of
#    the system.


# ----------------------------------------------------------------------------
# STEP 8: EARLY ATTEMPT - SIMPLY CHANGE TARGET TO BINARY EM (REJECTED)
# ----------------------------------------------------------------------------
# Suggested: replace soft-mean with binary EM (1 if all tokens right, else 0).
#
# User rightly pushed back: this target is too aggressive for sequence
# tasks. Until any sample achieves EM, target is always 0 and the Q-head
# has no learning signal at all. Gradient sparsity becomes the new failure.
#
# This pushed us toward needing a warmup phase OR a softer continuous
# target that still tracks EM more closely than soft-mean.


# ----------------------------------------------------------------------------
# STEP 9: TRIED OPTION 1 - THRESHOLD SHIFT (halt > 1.5 instead of halt > 0)
# ----------------------------------------------------------------------------
# Reasoning: if halt > 0 corresponds to sigmoid(h) > 0.5 (token accuracy
# 50%), then halt > 1.5 corresponds to sigmoid(h) > 0.82 (token accuracy
# 82%). Closer to EM. Should delay halt firing until model is much better.
#
# Result over 90 epochs:
#     Phase 1 (epochs 0-30): halt logits below 1.5, halt rarely fires,
#         model uses all 5 supervision steps, val accuracy slowly builds
#         (0.06% -> 13.9%).
#     Phase 2 (epochs 30-90): token accuracy gets high enough that target
#         saturates near 1, halt logits explode past 1.5, collapse re-emerges.
#         Val accuracy reaches 99.24% by epoch 90.
#
# Key observation: halt logits across supervision steps within ONE forward
# pass are nearly identical (e.g., 0.84, 0.86, 0.86, 0.86, 0.86). The Q-head
# is not discriminating by latent state - it outputs roughly the same logit
# regardless of where in the recursion we are. This confirms the latent
# state hits a fixed point fast on this task.


# ----------------------------------------------------------------------------
# STEP 10: WHY THRESHOLD SHIFT IS NOT A REAL FIX (ONLY A DELAY)
# ----------------------------------------------------------------------------
# As the model improves, token accuracy approaches 1.
# As token accuracy approaches 1, soft-mean target approaches 1.
# As target approaches 1, the equilibrium halt logit logit(t) approaches +infinity.
# Any fixed threshold gets crossed eventually.
#
# So threshold shift only buys time. The 30-epoch delay let us train with
# full recursion early on, and that produced a better final model than V1
# (99% vs ?, where V1 was last measured at 87% at epoch 40 - we never ran
# V1 to epoch 90 to get a fair comparison).
#
# The improvement might come from "early recursion as curriculum" rather
# than from the threshold shift directly. We cannot tell without running
# V1 to the same epoch count.


# ----------------------------------------------------------------------------
# STEP 11: TWO REAL FIXES STILL UNTESTED
# ----------------------------------------------------------------------------
#
# OPTION 2: GEOMETRIC MEAN OF PROBABILITIES
#     Target = product of per-token probabilities, then take L-th root.
#     Any low-probability token drags target down sharply. Closer to EM
#     in spirit but stays continuous, so gradient signal is preserved.
#     Note: geometric mean of binary indicators reduces to binary EM,
#     so the useful version uses the model's softmax probabilities.
#
# OPTION 3: BINARY EM WITH WARMUP
#     Target is 1 only when full sequence is correct, else 0. This matches
#     what halt > 0 should mean exactly.
#     Solves gradient sparsity by training CE only for the first N epochs,
#     then turning on Q-loss once train EM exceeds some threshold (e.g. 5%).
#     Use train EM (not val EM) because that determines whether Q-targets
#     are non-trivial during training.


# ----------------------------------------------------------------------------
# STEP 12: WHY WE DID NOT GO WITH PAPER'S TWO-HEAD BOOTSTRAP
# ----------------------------------------------------------------------------
# Paper uses Q_halt and Q_continue, where Q_continue is bootstrapped from
# next step's Q-values. This gives non-stationary targets that don't
# saturate as model improves.
#
# We rejected this because TRM's appeal is single-head simplicity.
# Adding a second head defeats the architectural premise. The interesting
# research question is: can we keep one head and find a target that works?


# ----------------------------------------------------------------------------
# CORE LESSON FROM ALL OF THIS
# ----------------------------------------------------------------------------
# The Q-loss bug is not "target value too lenient" or "stationary target"
# or "regime-dependent bistability". Those framings were tried and refined
# or falsified.
#
# The real bug is a SEMANTIC MISMATCH between target and decision rule:
#     Target (soft-mean)       = expected token accuracy
#     Decision rule (halt > 0) = "model has solved the problem"
# These two quantities are not the same thing.
#
# Any honest fix must either:
#     (a) Change target to match the rule (binary EM with warmup, or
#         geometric mean of probabilities)
#     (b) Change the decision rule to match the target (threshold shift,
#         which only delays the issue because the target saturates at 1)
#
# On 4-digit addition, the bug coexists with task simplicity, so even a
# perfect Q-loss only buys us bounded improvement. The fix needs to be
# tested on a task with real iterative depth (Sudoku, Maze) to show
# meaningful recursion utilization.


# ----------------------------------------------------------------------------
# QUICK MATH REFERENCE
# ----------------------------------------------------------------------------
# halt > 0  <=>  sigmoid(halt) > 0.5
# BCE-with-logits gradient: dL/dh = sigmoid(h) - t
# Optimizer pulls halt logit toward logit(t) = log(t / (1 - t))
# logit(0.5) = 0  -> this is the threshold-flipping point
# logit(0.95) = +2.94  -> halt > 3 fires only when target ~ 0.95
# As t -> 1, logit(t) -> +infinity, so any fixed threshold eventually crossed

In [ ]:
import torch
import torch.nn.functional as F
from collections import Counter

# --- Training Loop ---

def model_training_and_validation_with_mask(c, T, n_sup, HALT_THRESHOLD):
  model = TinyRModel(hidden_size, vocab_size, inp_seq_len=inp_seq_len, out_seq_len=out_seq_len).to(device)
  optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

  print('PAD_TOKEN: ', PAD_TOKEN)
  criterion = nn.CrossEntropyLoss(reduction='none', ignore_index=PAD_TOKEN)

  epochs = 100


  print('Training Logs ------ \n')
  for epoch in range(epochs):
        model.train()

        running_loss = 0
        running_q = 0
        running_n_taken = 0

        all_halt_steps = []
        all_halt_logits = []

        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            batch_size_curr = input_ids.size(0)
            seq_len = labels.size(1)

            active_mask = torch.ones(batch_size_curr, dtype=torch.bool, device=device)

            x = input_ids
            y = torch.zeros(batch_size_curr, hidden_size, device=device)
            z = torch.zeros(batch_size_curr, hidden_size, device=device)

            step_loss = step_q = 0
            overall_loss = torch.tensor(0.0, device=device)

            # at the start of each batch
            halt_step = torch.full((batch_size_curr,), n_sup, dtype=torch.long, device=device)


            for i in range(n_sup):
                f_out, halt, y_new, z_new = model(x, y, z, T, n=6)

                all_halt_logits.append(halt.squeeze().detach().cpu())

                # Update hidden states only for active samples
                y = torch.where(active_mask.unsqueeze(1), y_new.detach(), y)
                z = torch.where(active_mask.unsqueeze(1), z_new.detach(), z)

                seq_len = labels.size(1)  #this is padded to length 6 for now

                # Prediction Loss (Per Token: Batch * SeqLen)
                loss_per_token = criterion(f_out.view(-1, vocab_size), labels.view(-1))

                # Halting Loss (Per Sample: Batch)
                q_loss_per_sample = Q_LOSS(halt, f_out.view(-1, seq_len, vocab_size), labels)

                # Expand active mask to token level for prediction loss
                # active_mask: (B) -> (B, Seq) -> (B*Seq)


                token_mask = active_mask.unsqueeze(1).expand(-1, seq_len).reshape(-1).float()
                sample_mask = active_mask.float()

                num_active_tokens = token_mask.sum() + 1e-8
                num_active_samples = sample_mask.sum() + 1e-8

                # Weighted sum of losses for active samples
                current_step_loss = (loss_per_token * token_mask).sum() / num_active_tokens
                current_step_q = (q_loss_per_sample.squeeze() * sample_mask).sum() / num_active_samples

                overall_loss += (current_step_loss + c * current_step_q)

                step_loss += current_step_loss.item()
                step_q += current_step_q.item()

                # Update halting decision
                halt_decision = (halt.squeeze() > HALT_THRESHOLD) & active_mask

                # inside the for i in range(n_sup): loop, after halt_decision is computed
                halt_step[halt_decision] = i + 1

                active_mask = active_mask & (~halt_decision)

                if active_mask.sum() == 0:
                  break

            # at the end of the batch, accumulate
            all_halt_steps.append(halt_step.cpu())

            n_taken = i + 1
            overall_loss /= n_taken
            # inside the step loop, after `n_taken = i + 1`:
            running_n_taken += n_taken

            optimizer.zero_grad()
            overall_loss.backward()
            optimizer.step()

            running_loss += step_loss / n_taken
            running_q += step_q / n_taken



        avg_loss = running_loss / len(train_loader)
        avg_q_loss = running_q / len(train_loader)

        avg_steps = running_n_taken / len(train_loader)

        print(f'Training processed epoch {epoch}')
        if epoch % 10 == 0:
            print(f" Training epoch {epoch} | CE: {avg_loss:.4f} | Q-Loss: {avg_q_loss:.4f} | Avg Steps: {avg_steps}")

            all_halt_steps = torch.cat(all_halt_steps)
            all_halt_logits = torch.cat(all_halt_logits)

            mean_h = all_halt_steps.float().mean().item()
            median_h = all_halt_steps.float().median().item()
            frac_1 = (all_halt_steps == 1).float().mean().item()
            frac_max = (all_halt_steps == n_sup).float().mean().item()
            dist = dict(Counter(all_halt_steps.tolist()))

            print(f"halt logit: mean={all_halt_logits.mean().item():.3f} "
                  f"std={all_halt_logits.std().item():.3f} "
                  f"max={all_halt_logits.max().item():.3f} "
                  f"%>0={(all_halt_logits > 0).float().mean().item():.1%} "
                  f"%>1.5={(all_halt_logits > 1.5).float().mean().item():.1%}")

            print(f"halt steps: mean={mean_h:.2f} median={median_h:.0f} "
                  f"%step1={frac_1:.1%} %never={frac_max:.1%} dist={dist}")

        running_loss = running_q = 0


        if epoch % 10 == 0:
            print('\nValidation Logs ------ \n')
            model.eval()
            correct = 0
            total = 0

            all_halt_steps_validation = []

            with torch.no_grad():
                for val_step, batch in enumerate(val_loader):
                    input_ids = batch['input_ids'].to(device)
                    labels = batch['labels'].to(device)
                    batch_size_curr = input_ids.size(0)

                    active_mask = torch.ones(batch_size_curr, dtype=torch.bool, device=device)

                    # at the start of each batch
                    val_halt_step = torch.full((batch_size_curr,), n_sup, dtype=torch.long, device=device)

                    x = input_ids
                    y = torch.zeros(input_ids.size(0), hidden_size, device=device)
                    z = torch.zeros(input_ids.size(0), hidden_size, device=device)

                    final_logits = torch.zeros(batch_size_curr, out_seq_len * vocab_size, device=device)

                    for i in range(n_sup):
                        f_out_val, val_halt, y_new, z_new = model(x, y, z, T, n=6)
                        y, z = y_new.detach(), z_new.detach()

                        # Update halting decision
                        if val_step == 0:
                            print(f"sup_step_{i+1} | mean={val_halt.mean().item():.3f} "
                                  f"std={val_halt.std().item():.3f} "
                                  f"median={val_halt.median().item():.3f} "
                                  f"%>0={(val_halt > 0).float().mean().item():.1%} "
                                  f"%>1.5={(val_halt > 1.5).float().mean().item():.1%} "
                                  f"first5={val_halt[:5].squeeze().tolist()}")

                        halt_decision = (val_halt.squeeze() > HALT_THRESHOLD) & active_mask
                               # inside the for i in range(n_sup): loop, after halt_decision is computed
                        val_halt_step[halt_decision] = i + 1

                        active_mask = active_mask & (~halt_decision)

                        final_logits[halt_decision] = f_out_val[halt_decision]

                        if active_mask.sum() == 0:
                           break

                    # at the end of the batch, accumulate
                    all_halt_steps_validation.append(val_halt_step.cpu())

                    final_logits[active_mask] = f_out_val[active_mask]

                    preds = final_logits.view(-1, seq_len, vocab_size).argmax(dim=-1)
                    non_pad = (labels != PAD_TOKEN)
                    matches_per_token = (preds == labels) | ~non_pad
                    exact = matches_per_token.all(dim=-1)

                    correct += exact.sum().item()
                    total   += labels.size(0)


            print(f"\nValidation Accuracy: {100 * correct / total:.2f}%")

            all_halt_steps_validation = torch.cat(all_halt_steps_validation)

            mean_h = all_halt_steps_validation.float().mean().item()
            median_h = all_halt_steps_validation.float().median().item()
            frac_1 = (all_halt_steps_validation == 1).float().mean().item()
            frac_max = (all_halt_steps_validation == n_sup).float().mean().item()
            dist = dict(Counter(all_halt_steps_validation.tolist()))

            print(f"validation halt steps: mean={mean_h:.2f} median={median_h:.0f} "
                  f"%step1={frac_1:.1%} %never={frac_max:.1%} dist={dist}")

In [ ]:
import torch
import torch.nn.functional as F

def model_training_and_validation(c, T, n_sup):
    running_loss = 0
    running_q = 0

    # Use PAD_TOKEN for ignoring index in loss
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN)

    # Initialize model with correct sequence lengths
    model = TinyRModel(hidden_size, vocab_size, inp_seq_len, out_seq_len).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    epochs = 100

    print('Training Logs ------ \n')
    for _ in range(epochs):
        model.train()
        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)

            x = input_ids
            # Initial states must be (Batch, hidden_size)
            y = torch.zeros(input_ids.size(0), hidden_size, device=device)
            z = torch.zeros(input_ids.size(0), hidden_size, device=device)

            step_loss = step_q = 0
            overall_loss = 0
            n_taken = 0

            for i in range(n_sup):
                n_taken = i + 1
                f_out, halt, y_new, z_new = model(x, y, z, T, n=6)
                y, z = y_new.detach(), z_new.detach()

                # Map f_out (B, out_seq_len * vocab_size) to (B * out_seq_len, vocab_size)
                seq_len = labels.size(1)

                logits = f_out.view(-1, vocab_size)
                targets = labels.view(-1)

                # print(halt.shape, targets.shape, labels.shape, logits.shape)

                loss = criterion(logits, targets)

                # print(f_out.shape, labels.shape)
                q_loss_val = Q_LOSS(halt, f_out.view(-1, seq_len, vocab_size), labels)

                overall_loss += (loss + c * q_loss_val)
                step_loss += loss.item()
                step_q += q_loss_val.item()

                if (halt > 0).all():
                    break

            overall_loss = overall_loss / n_taken
            optimizer.zero_grad()
            overall_loss.backward()
            optimizer.step()

            running_loss += step_loss / n_taken
            running_q += step_q / n_taken

            if step % 50 == 0 and step > 0 and n_taken<n_sup:
                print(f"Step {step} | CE: {running_loss/50:.4f} | Q-Loss: {running_q/50:.4f} | Avg Steps: {n_taken}")
                running_loss = running_q = 0

        print('\nValidation Logs ------ \n')
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for val_step, batch in enumerate(val_loader):
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)
                x = input_ids
                y = torch.zeros(input_ids.size(0), hidden_size, device=device)
                z = torch.zeros(input_ids.size(0), hidden_size, device=device)

                for i in range(n_sup):
                    f_out_val, val_halt, y_new, z_new = model(x, y, z, T, n=6)
                    y, z = y_new.detach(), z_new.detach()
                    if (val_halt > 0).all(): break

                logits = f_out_val.view(-1, vocab_size)
                targets = labels.view(-1)
                predictions = logits.argmax(dim=1)

                mask = targets != PAD_TOKEN
                correct += (predictions[mask] == targets[mask]).sum().item()
                total += mask.sum().item()

        if epoch % 10 == 0:
            print(f"\nValidation Accuracy: {100 * correct / total:.2f}%")

In [ ]:
# Accessing the underlying .data list from the AdditionDataset object
# Start with 2-digit additions with no carries
easy_dataset = [d for d in dataset.data if d['num_digits'] <= 2 and d['max_carry_chain'] == 0]
print(f"Easy examples: {len(easy_dataset)}")

# If that works, try with carries
medium_dataset = [d for d in dataset.data if d['num_digits'] <= 2]
print(f"Medium examples: {len(medium_dataset)}")

# Then scale up
hard_dataset = [d for d in dataset.data if d['num_digits'] == 4]
print(f"Hard examples: {len(hard_dataset)}")

Easy examples: 0
Medium examples: 0
Hard examples: 50000


In [ ]:
## t-0

# Config: n_sup=5, halt > 0, c=0.1, T=3, 100 epochs
# Target = exp(-mean CE), so halt > 0 fires when train mean CE crosses ln 2 ~ 0.693.
# Prediction: collapse onset readable straight off the CE curve; later than soft-mean's 50% token-accuracy crossing.
# Log: training-logs/geomean-halt-0.txt

HALT_THRESHOLD = 0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=5, HALT_THRESHOLD = HALT_THRESHOLD)


PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6453 | Q-Loss: 0.4628 | Avg Steps: 5.0
halt logit: mean=-1.558 std=0.379 max=-0.019 %>0=0.0% %>1.5=0.0%
halt steps: mean=5.00 median=5 %step1=0.0% %never=100.0% dist={5: 45000}

Validation Logs ------ 

sup_step_1 | mean=-1.451 std=0.258 median=-1.412 %>0=0.0% %>1.5=0.0% first5=[-1.4119447469711304, -1.4216440916061401, -1.7959693670272827, -1.615000605583191, -1.152976155281067]
sup_step_2 | mean=-1.479 std=0.283 median=-1.400 %>0=0.0% %>1.5=0.0% first5=[-1.3521850109100342, -1.3303831815719604, -1.5891951322555542, -2.008553981781006, -1.150156855583191]
sup_step_3 | mean=-1.445 std=0.242 median=-1.377 %>0=0.0% %>1.5=0.0% first5=[-1.3210846185684204, -1.3122841119766235, -1.6144052743911743, -1.7882157564163208, -1.1103039979934692]
sup_step_4 | mean=-1.462 std=0.266 median=-1.390 %>0=0.0% %>1.5=0.0% first5=[-1.3244737386703491, -1.3161402940750122, -1.6108256578445435, -1.9229401350021362, -1.

In [ ]:
## t-1.5

# Config: n_sup=5, halt > 1.5, c=0.1, T=3, 100 epochs
# halt > 1.5 fires when target > sigmoid(1.5) ~ 0.818, i.e. mean CE < 0.201.
# Log: training-logs/geomean-halt-1.5.txt

HALT_THRESHOLD = 1.5
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=5, HALT_THRESHOLD = HALT_THRESHOLD)


PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6364 | Q-Loss: 0.4645 | Avg Steps: 5.0
halt logit: mean=-1.550 std=0.376 max=-0.024 %>0=0.0% %>1.5=0.0%
halt steps: mean=5.00 median=5 %step1=0.0% %never=100.0% dist={5: 45000}

Validation Logs ------ 

sup_step_1 | mean=-1.372 std=0.240 median=-1.285 %>0=0.0% %>1.5=0.0% first5=[-1.3642785549163818, -1.26469886302948, -1.6628265380859375, -1.6224404573440552, -1.2789182662963867]
sup_step_2 | mean=-1.395 std=0.263 median=-1.316 %>0=0.0% %>1.5=0.0% first5=[-1.3065173625946045, -1.210601806640625, -1.698033094406128, -1.6226410865783691, -1.1652313470840454]
sup_step_3 | mean=-1.412 std=0.261 median=-1.317 %>0=0.0% %>1.5=0.0% first5=[-1.345550298690796, -1.2388193607330322, -1.7029423713684082, -1.6258454322814941, -1.1676340103149414]
sup_step_4 | mean=-1.409 std=0.265 median=-1.324 %>0=0.0% %>1.5=0.0% first5=[-1.3458731174468994, -1.2408537864685059, -1.7043286561965942, -1.6278363466262817, -1.1

In [ ]:
## t-3.0

# Config: n_sup=5, halt > 3.0, c=0.1, T=3, 100 epochs
# halt > 3.0 fires when target > sigmoid(3.0) ~ 0.953, i.e. mean CE < 0.049.
# Prediction: longest delay of the three, still collapses as CE -> 0 (target -> 1, equilibrium logit -> +inf).
# Log: training-logs/geomean-halt-3.0.txt

HALT_THRESHOLD = 3.0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=5, HALT_THRESHOLD = HALT_THRESHOLD)


PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6290 | Q-Loss: 0.4681 | Avg Steps: 5.0
halt logit: mean=-1.543 std=0.400 max=0.008 %>0=0.0% %>1.5=0.0%
halt steps: mean=5.00 median=5 %step1=0.0% %never=100.0% dist={5: 45000}

Validation Logs ------ 

sup_step_1 | mean=-1.366 std=0.340 median=-1.223 %>0=0.0% %>1.5=0.0% first5=[-1.31161367893219, -1.1193115711212158, -1.6073282957077026, -1.92433762550354, -1.1861685514450073]
sup_step_2 | mean=-1.269 std=0.239 median=-1.180 %>0=0.0% %>1.5=0.0% first5=[-1.169736385345459, -0.9771987795829773, -1.596959114074707, -1.5175073146820068, -1.238845944404602]
sup_step_3 | mean=-1.297 std=0.269 median=-1.209 %>0=0.0% %>1.5=0.0% first5=[-1.2088648080825806, -1.0038665533065796, -1.59055495262146, -1.6606818437576294, -1.2425644397735596]
sup_step_4 | mean=-1.288 std=0.255 median=-1.211 %>0=0.0% %>1.5=0.0% first5=[-1.2181756496429443, -1.0145982503890991, -1.5962320566177368, -1.5997172594070435, -1.241285

In [ ]:
## s-1

# Config: n_sup=1, halt > 0, c=0.1, T=3, 100 epochs
# Supervision sweep: no-recursion baseline under the geo-mean target.
# Log: training-logs/geomean-nsup-1.txt

HALT_THRESHOLD = 0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=1, HALT_THRESHOLD = HALT_THRESHOLD)


PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6230 | Q-Loss: 0.4684 | Avg Steps: 1.0
halt logit: mean=-1.540 std=0.371 max=-0.016 %>0=0.0% %>1.5=0.0%
halt steps: mean=1.00 median=1 %step1=100.0% %never=100.0% dist={1: 45000}

Validation Logs ------ 

sup_step_1 | mean=-1.386 std=0.265 median=-1.273 %>0=0.0% %>1.5=0.0% first5=[-1.1327728033065796, -1.268949031829834, -1.6784000396728516, -1.7189247608184814, -1.2018095254898071]

Validation Accuracy: 0.04%
validation halt steps: mean=1.00 median=1 %step1=100.0% %never=100.0% dist={1: 5000}
Training processed epoch 1
Training processed epoch 2
Training processed epoch 3
Training processed epoch 4
Training processed epoch 5
Training processed epoch 6
Training processed epoch 7
Training processed epoch 8
Training processed epoch 9
Training processed epoch 10
 Training epoch 10 | CE: 0.6366 | Q-Loss: 0.6916 | Avg Steps: 1.0
halt logit: mean=0.034 std=0.208 max=0.874 %>0=56.2% %>1.5=0.0%
halt step

In [ ]:
## s-16

# Config: n_sup=16, halt > 0, c=0.1, T=3, 100 epochs
# Supervision sweep at the paper's N_sup = 16.
# Slow early: target < 0.5 while CE > 0.693, so no halting -> all 16 steps run per batch.
# Log: training-logs/geomean-nsup-16.txt

HALT_THRESHOLD = 0
model_training_and_validation_with_mask(c=0.1, T=3, n_sup=16, HALT_THRESHOLD = HALT_THRESHOLD)


PAD_TOKEN:  11
Training Logs ------ 

Training processed epoch 0
 Training epoch 0 | CE: 1.6240 | Q-Loss: 0.4683 | Avg Steps: 16.0
halt logit: mean=-1.540 std=0.386 max=-0.031 %>0=0.0% %>1.5=0.0%
halt steps: mean=16.00 median=16 %step1=0.0% %never=100.0% dist={16: 45000}

Validation Logs ------ 

sup_step_1 | mean=-1.454 std=0.302 median=-1.449 %>0=0.0% %>1.5=0.0% first5=[-1.173067569732666, -1.0509393215179443, -1.9469919204711914, -1.8222236633300781, -1.2033684253692627]
sup_step_2 | mean=-1.426 std=0.263 median=-1.316 %>0=0.0% %>1.5=0.0% first5=[-1.27543044090271, -1.1189897060394287, -1.9054815769195557, -1.777728796005249, -1.3159399032592773]
sup_step_3 | mean=-1.450 std=0.256 median=-1.385 %>0=0.0% %>1.5=0.0% first5=[-1.2607438564300537, -1.1124589443206787, -1.8956336975097656, -1.808718204498291, -1.3209147453308105]
sup_step_4 | mean=-1.445 std=0.258 median=-1.391 %>0=0.0% %>1.5=0.0% first5=[-1.2657902240753174, -1.1144130229949951, -1.8997583389282227, -1.8032784461975098, 

## Key Observations

- Using mean loss across steps harms per-example optimization
- If q_halt becomes negative early, future optimization is skipped
- Masking improves stability by preserving active samples


### Key Observation: Effect of Halting Coefficient (c)

- c controls the trade-off between prediction and halting behavior
- With small c (0.01), model halts after 1 step (n_taken = 1)
- This reduces effective reasoning depth despite high accuracy
- Indicates that halting must be carefully balanced to enable multi-step reasoning